# Review export: small video pieces + compact tracks for the pass review page

Cuts the match into 1-minute, 960-pixel-wide video pieces (each under 4.5 MB, so Claude can fetch them
through the Drive connector) and writes a compact tracks file per 5-minute segment. Claude then
proposes the passes and publishes the review page you use on the tablet; no Colab while reviewing.

Output: `MyDrive/Playbook/review/match_video_2/<segment>/` (`part_00.mp4` ..., `tracks_small.csv.gz`).
**SEGMENTS** below says which segments; the pilot is the first one.

**Runtime: CPU is enough** (Runtime -> Change runtime type -> CPU), about 2-3 minutes per segment.
**Runtime -> Run all**; do not edit cells. Finished segments are skipped.

In [ ]:
# Cell 1 — Drive and inputs
import os, json, glob, shutil, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
PBD = '/content/drive/MyDrive/Playbook'
MATCH = 'match_video_2'
SEGMENTS = ['01_1H_00-05']          # pilot; later: all segments
FULL = f'{PBD}/full_match/{MATCH}'
CLIPS = f'{PBD}/video_downloads/{MATCH}_clips'
OUT = f'{PBD}/review/{MATCH}'
for p in (f'{FULL}/match_tracks.csv.gz', *[f'{CLIPS}/{s}.mp4' for s in SEGMENTS]):
    if not os.path.exists(p):
        raise SystemExit(f'Missing in Drive: {p}')
print('inputs found')

In [ ]:
# Cell 2 — Compact tracks per segment (match frames, so names and joined players match the full match)
import pandas as pd, numpy as np
COLS = ['frame', 'segment', 'segment_frame', 'raw_tracker_id', 'joined_id', 'class_id', 'team_id',
        'x1', 'y1', 'x2', 'y2', 'x_m', 'y_m']
todo = [s for s in SEGMENTS if not os.path.exists(f'{OUT}/{s}/tracks_small.csv.gz')]
if todo:
    t = pd.read_csv(f'{FULL}/match_tracks.csv.gz', low_memory=False, usecols=lambda c: c in COLS)
    for s in todo:
        g = t[t.segment == s].drop(columns='segment').copy()
        for c in ('x1', 'y1', 'x2', 'y2'):
            g[c] = g[c].round().astype(int)
        for c in ('x_m', 'y_m'):
            g[c] = g[c].round()
        os.makedirs(f'{OUT}/{s}', exist_ok=True)
        g.to_csv(f'{OUT}/{s}/tracks_small.csv.gz', index=False, compression='gzip')
        print(s, len(g), 'rows', f"{os.path.getsize(f'{OUT}/{s}/tracks_small.csv.gz') / 1e6:.1f} MB")
    del t
print('tracks done')

In [ ]:
# Cell 3 — 1-minute video pieces, 960 px wide, each under 4.5 MB (exact frames: part k = frames 1800k ...)
import cv2
def frames_in(p):
    c = cv2.VideoCapture(p); n = int(c.get(cv2.CAP_PROP_FRAME_COUNT)); f = c.get(cv2.CAP_PROP_FPS); c.release(); return n, f
for s in SEGMENTS:
    src = f'{CLIPS}/{s}.mp4'
    n, fps = frames_in(src)
    per = int(round(60 * fps))
    local = f'/content/{s}.mp4'
    if not os.path.exists(local):
        shutil.copy(src, local)
    info = {'segment': s, 'fps': fps, 'frames': n, 'frames_per_part': per, 'width': 960, 'parts': []}
    for k in range((n + per - 1) // per):
        dst = f'{OUT}/{s}/part_{k:02d}.mp4'
        a, b = k * per, min(n, (k + 1) * per)
        if not os.path.exists(dst):
            for crf in (30, 33, 36, 39):
                vf = f"select='between(n,{a},{b - 1})',setpts=N/FRAME_RATE/TB,scale=960:-2"
                r = subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', local, '-vf', vf, '-an',
                                    '-c:v', 'libx264', '-preset', 'veryfast', '-crf', str(crf), '-g', '30',
                                    '-pix_fmt', 'yuv420p', '-movflags', '+faststart', '-vsync', '0', '/content/part.mp4'],
                                   capture_output=True, text=True)
                if r.returncode != 0:
                    raise SystemExit(r.stderr[-1500:])
                if os.path.getsize('/content/part.mp4') < 4.5e6:
                    break
            shutil.copy('/content/part.mp4', dst)
        m, _ = frames_in(dst)
        info['parts'].append({'file': os.path.basename(dst), 'first_frame': a, 'frames': m,
                              'mb': round(os.path.getsize(dst) / 1e6, 2)})
        print(s, os.path.basename(dst), m, 'frames (expected', b - a, ')', info['parts'][-1]['mb'], 'MB')
    json.dump(info, open(f'{OUT}/{s}/parts.json', 'w'), indent=1)
print(f'\nDone. Tell Claude the review export is in {OUT}.')